# HeatShield AI — Colab Training

**กดรัน Cell เดียวแล้วรอ** — ได้ ZIP โมเดลใน Google Drive อัตโนมัติ

**ตั้งค่าก่อนรัน (ไม่บังคับ):**
- Colab Secrets → `CDSAPI_KEY`, `TMD_API_KEY` (ถ้าอยากใช้ ERA5/TMD)
- Runtime → Change runtime type → GPU (T4) เพื่อความเร็ว

**Output:** ZIP อยู่ที่ `MyDrive/heatshield/exports/HeatShield_artifacts_v*.zip`

In [ ]:
# ============================================================
# CONFIG — แก้ตรงนี้ถ้าต้องการ
# ============================================================
STAGE1_TRIALS = 60    # trials สำหรับ h=24 sweep
STAGE2_TRIALS = 150   # trials สำหรับ weak slots
STAGE3_TRIALS = 100   # trials สำหรับ h=6/12 (ถ้าเปิดใช้)
RUN_STAGE3    = True  # False = ข้าม h=6/12, เทรนแค่ h=24
START_DATE    = "2021-01-01"
MODEL_VERSION = "v4"  # Change to v5, v6, etc. for new training runs
# ============================================================

import datetime, hashlib, importlib, json, os, shutil, subprocess, sys, time, zipfile
from collections import defaultdict
from pathlib import Path

# ── secrets ────────────────────────────────────────────────
try:
    from google.colab import drive, userdata
    for _k in ("CDSAPI_KEY", "TMD_API_KEY"):
        try: os.environ[_k] = userdata.get(_k)
        except Exception: pass
    _ON_COLAB = True
except ImportError:
    _ON_COLAB = False

# ── Drive mount ────────────────────────────────────────────
if _ON_COLAB:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/heatshield")
else:
    DRIVE_ROOT = Path("/tmp/heatshield")

MODEL_DIR    = DRIVE_ROOT / "models" / f"forecast_{MODEL_VERSION}"
VERSIONS_DIR = DRIVE_ROOT / "models" / "forecast_versions"
EXPORTS_DIR  = DRIVE_ROOT / "exports"
for _d in (MODEL_DIR, VERSIONS_DIR, EXPORTS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# ── clone repo ─────────────────────────────────────────────
REPO      = "/content/Heat-wave-backend"
GH_OWNER  = "orbitorls"
GH_REPO   = "HeatShield"
GH_BRANCH = "main"

if not Path(f"{REPO}/.git").is_dir():
    print("Cloning repo ...")
    subprocess.run(
        ["git", "clone", "--depth=1", f"--branch={GH_BRANCH}",
         f"https://github.com/{GH_OWNER}/{GH_REPO}.git", REPO],
        check=True,
    )
else:
    print("Updating repo ...")
    subprocess.run(["git", "-C", REPO, "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", f"origin/{GH_BRANCH}"], check=True)

# ── install deps ───────────────────────────────────────────
REQ       = f"{REPO}/requirements-train.txt"
HASH_FILE = f"{REPO}/.colab_req_hash"
new_hash  = hashlib.sha256(Path(REQ).read_bytes()).hexdigest()
old_hash  = Path(HASH_FILE).read_text().strip() if Path(HASH_FILE).exists() else ""
if new_hash != old_hash:
    print("Installing deps ...")
    subprocess.run(["pip", "install", "-q", "-r", REQ], check=True)
    Path(HASH_FILE).write_text(new_hash)
else:
    print("Deps unchanged — skip install")

# ── python path ────────────────────────────────────────────
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

# ── symlink model dir → Drive ──────────────────────────────
LOCAL_MODELS = Path(REPO) / "app" / "models" / f"forecast_{MODEL_VERSION}"
LOCAL_MODELS.parent.mkdir(parents=True, exist_ok=True)
if LOCAL_MODELS.exists() and not LOCAL_MODELS.is_symlink():
    for _item in LOCAL_MODELS.iterdir():
        _t = MODEL_DIR / _item.name
        if not _t.exists():
            shutil.move(str(_item), str(_t))
    shutil.rmtree(str(LOCAL_MODELS))
if not LOCAL_MODELS.exists():
    LOCAL_MODELS.symlink_to(str(MODEL_DIR))

# ── detect device ──────────────────────────────────────────
def _detect_device() -> str:
    """Return 'gpu' (OpenCL) if Colab has GPU, else 'cpu'."""
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=5,
        )
        if result.returncode == 0 and result.stdout.strip():
            gpu_name = result.stdout.strip().splitlines()[0]
            print(f"GPU detected: {gpu_name} → device=gpu (OpenCL)")
            return "gpu"
    except Exception:
        pass
    print("No GPU detected → device=cpu")
    return "cpu"

DEVICE = _detect_device()
# CPU: ใช้หลาย workers เพื่อ parallel แทน GPU
N_WORKERS = 1 if DEVICE == "gpu" else max(2, min(4, (os.cpu_count() or 2) // 2))
print(f"Device={DEVICE}, Workers={N_WORKERS}")

# ── validate data ──────────────────────────────────────────
END_DATE = datetime.date.today().isoformat()
print(f"\nIngesting NASA POWER {START_DATE} → {END_DATE} ...")
subprocess.run(
    ["python", "scripts/ingest_nasa_power.py", "--start", START_DATE, "--end", END_DATE],
    cwd=REPO, check=True,
)

from datetime import date as _date
from app.data.stations import STATIONS
from app.data.loaders import read_observations

print("\nData check:")
for sid in STATIONS:
    n = len(read_observations(sid, _date.fromisoformat(START_DATE), _date.fromisoformat(END_DATE)))
    print(f"  {sid}: {n} rows")
    if n < 500:
        raise RuntimeError(f"Not enough data for {sid}: {n} rows")
print("Data OK")

# ── training helper ────────────────────────────────────────
LOG_DIR   = Path(REPO) / "logs" / "train"
RUNS_ROOT = Path(REPO) / "logs" / "eval" / "runs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

RUN_IDS: list[str] = []
_RID_SET: set[str] = set()
TIMINGS: list[dict] = []


def _train(tag, *, trials, horizons=None, station=None, force=False, model_version=None):
    run_id = f"{tag}_{datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
    cmd = [
        "python", "-u", "scripts/train_forecast.py",
        "--trials", str(trials), "--start", START_DATE, "--end", END_DATE,
    ]
    if "--run-id" in _help: cmd += ["--run-id", run_id]
    if "--device" in _help: cmd += ["--device", DEVICE]
    if "--gate-backend" in _help: cmd += ["--gate-backend", "lightgbm"]
    if "--workers" in _help and N_WORKERS > 1: cmd += ["--workers", str(N_WORKERS)]
    # Always add --model-version if provided (bypass help check for Colab compatibility)
    if model_version: cmd += ["--model-version", model_version]
    if horizons: cmd += ["--horizons", horizons]
    if station:  cmd += ["--station",  station]
    if force and "--force" in _help: cmd += ["--force"]

    log_path = LOG_DIR / f"{run_id}.log"
    print(f"\n{'='*60}")
    print(f"RUN: {run_id}")
    print(f"CMD: {' '.join(cmd)}")
    print(f"{'='*60}")

    t0  = time.perf_counter()
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    with open(log_path, "w", encoding="utf-8") as _f:
        _p = subprocess.Popen(
            cmd, cwd=REPO,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env=env,
        )
        for line in _p.stdout:
            print(line, end="")
            _f.write(line)
        rc = _p.wait()

    elapsed = time.perf_counter() - t0
    if rc != 0:
        raise RuntimeError(f"Training failed rc={rc} — {log_path}")
    if run_id not in _RID_SET:
        _RID_SET.add(run_id)
        RUN_IDS.append(run_id)
    TIMINGS.append({"tag": tag, "seconds": round(elapsed, 2)})
    print(f"\nDone in {elapsed/60:.1f} min")
    return run_id


# ── STAGE 1: h=24 sweep ────────────────────────────────────
print("\n" + "#"*60)
print("# STAGE 1 — h=24 sweep (all stations)")
print("#"*60)
s1_id = _train("stage1", trials=STAGE1_TRIALS, horizons="24", model_version=MODEL_VERSION)

# ── STAGE 2: refine weak slots ─────────────────────────────
print("\n" + "#"*60)
print("# STAGE 2 — refine weak slots")
print("#"*60)
lb1 = RUNS_ROOT / s1_id / "leaderboard.json"
if lb1.exists():
    weak_by_station: dict[str, set[int]] = defaultdict(set)
    for r in json.loads(lb1.read_text()):
        status = str(r.get("status", "")).lower()
        if status in {"not_ready", "candidate"} or r.get("skill_score") is None:
            sid, h = r.get("station"), r.get("horizon_h")
            if sid and h is not None:
                weak_by_station[str(sid)].add(int(h))
    print(f"Weak slots: {sum(len(v) for v in weak_by_station.values())}")
    for sid in sorted(weak_by_station):
        hz = ",".join(str(h) for h in sorted(weak_by_station[sid]))
        _train(f"stage2_{sid}", trials=STAGE2_TRIALS, station=sid, horizons=hz, force=True, model_version=MODEL_VERSION)
else:
    print("[warn] No leaderboard from Stage 1 — skipping Stage 2")

# ── STAGE 3: h=6/12 (optional) ────────────────────────────
if RUN_STAGE3:
    print("\n" + "#"*60)
    print("# STAGE 3 — h=6/12 for ready stations")
    print("#"*60)
    all_rows: list[dict] = []
    for rid in RUN_IDS:
        lb = RUNS_ROOT / rid / "leaderboard.json"
        if lb.exists():
            all_rows.extend(json.loads(lb.read_text()))
    latest: dict = {}
    for r in all_rows:
        latest[(r.get("station"), r.get("horizon_h"))] = r
    ready = sorted({
        sid for (sid, h), r in latest.items()
        if h == 24 and str(r.get("status", "")).lower() == "ready"
    })
    print(f"Ready stations: {ready}")
    for sid in ready:
        _train(f"stage3_{sid}", trials=STAGE3_TRIALS, station=sid, horizons="6,12", model_version=MODEL_VERSION)

# ── EXPORT ─────────────────────────────────────────────────
print("\n" + "#"*60)
print("# EXPORT — building ZIP snapshot")
print("#"*60)

def _next_ver(base: Path) -> int:
    vs = [int(d.name[1:]) for d in base.iterdir()
          if d.is_dir() and d.name.startswith("v") and d.name[1:].isdigit()]
    return (max(vs) + 1) if vs else 1

slot_meta: dict = {}
warnings: list[str] = []
for rid in RUN_IDS:
    lb = RUNS_ROOT / rid / "leaderboard.json"
    if not lb.exists():
        warnings.append(f"missing_lb:{rid}")
        continue
    for r in json.loads(lb.read_text()):
        sid, h = r.get("station"), r.get("horizon_h")
        if sid in (None, "all") or h is None: continue
        slot_meta[(str(sid), int(h))] = {
            "station": str(sid), "horizon_h": int(h),
            "backend": r.get("backend"), "status": r.get("status"),
            "source_run_id": rid,
        }

snap_ver = f"v{_next_ver(VERSIONS_DIR)}"
snap_dir = VERSIONS_DIR / snap_ver
snap_dir.mkdir(parents=True, exist_ok=False)

for (sid, h) in sorted(slot_meta):
    src = LOCAL_MODELS / sid / f"h{h}"
    dst = snap_dir / sid / f"h{h}"
    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        warnings.append(f"missing:{sid}:h{h}")

cm = LOCAL_MODELS / "choice_matrix.json"
if cm.exists(): shutil.copy2(cm, snap_dir / "choice_matrix.json")

manifest = {
    "created_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "snapshot_version": snap_ver,
    "model_version": MODEL_VERSION,
    "run_ids": RUN_IDS,
    "slots": [slot_meta[k] for k in sorted(slot_meta)],
    "warnings": warnings,
    "timings": TIMINGS,
}
(snap_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))

stamp    = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
zip_name = f"HeatShield_artifacts_{snap_ver}_{stamp}.zip"
zip_path = EXPORTS_DIR / zip_name

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for fp in sorted(snap_dir.rglob("*")):
        if fp.is_file():
            zf.write(fp, (Path("models") / snap_ver / fp.relative_to(snap_dir)).as_posix())
    for rid in RUN_IDS:
        run_dir = RUNS_ROOT / rid
        if run_dir.exists():
            for fp in sorted(run_dir.rglob("*")):
                if fp.is_file():
                    zf.write(fp, (Path("eval") / rid / fp.relative_to(run_dir)).as_posix())
    zf.writestr("manifest.json", json.dumps(manifest, indent=2, ensure_ascii=False))

mb = zip_path.stat().st_size / 1024 / 1024
total_min = sum(t["seconds"] for t in TIMINGS) / 60

print(f"""
{'='*60}
DONE
  Snapshot : {snap_ver}
  Model Ver: {MODEL_VERSION}
  Slots    : {len(slot_meta)}
  ZIP      : {zip_path}
  Size     : {mb:.1f} MB
  Total    : {total_min:.0f} min
{'='*60}

ดาวน์โหลด ZIP จาก Google Drive:
  MyDrive/heatshield/exports/{zip_name}
""")

if warnings:
    print("Warnings:")
    for w in warnings:
        print(" -", w)

## วิธีใช้โมเดล (Windows)

```powershell
# 1. Download ZIP จาก MyDrive/heatshield/exports/

# 2. Extract + promote
$ZIP = Get-ChildItem .\HeatShield_artifacts_v*.zip | Sort LastWriteTime -Desc | Select -First 1
Expand-Archive $ZIP.FullName -DestinationPath .\artifact_unpack -Force
$VER = Get-ChildItem ".\artifact_unpack\models" -Directory | Sort Name -Desc | Select -First 1
Remove-Item app\models\forecast_v3 -Recurse -Force -ErrorAction SilentlyContinue
New-Item -ItemType Directory -Force -Path app\models\forecast_v3 | Out-Null
Copy-Item "$($VER.FullName)\*" "app\models\forecast_v3\" -Recurse -Force

# 3. Verify + evaluate
python -c "from app.ml.registry import load_latest_v3; print(load_latest_v3('BKK_01', 24).backend_name)"
python scripts/evaluate_model.py
```